# agent-desktop 사용법: 접근성 트리로 데스크톱 앱을 읽고 클릭하는 에이전트 CLI, 스크린샷 방식 Computer Use와 뭐가 다른지와 snapshot→click→snapshot 첫 실행

위에서부터 차례로 실행해 보세요.

### 패키지와 agent-desktop 설치

In [ ]:
%%bash
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xvfb at-spi2-core dbus-x11 gnome-calculator imagemagick openbox x11-apps
npm install -g agent-desktop
agent-desktop version

### 가상 디스플레이·D-Bus 세션·계산기 띄우기

In [ ]:
%%bash
cd /work 2>/dev/null || mkdir -p /work && cd /work
cat > start.sh <<'E'
export DISPLAY=:99
setsid Xvfb :99 -screen 0 1280x800x24 >/tmp/xvfb.log 2>&1 &
sleep 2
setsid openbox >/tmp/ob.log 2>&1 &
eval $(dbus-launch --sh-syntax)
export GTK_MODULES=gail:atk-bridge GNOME_ACCESSIBILITY=1
echo "export DISPLAY=:99 DBUS_SESSION_BUS_ADDRESS='$DBUS_SESSION_BUS_ADDRESS'" > /work/envfile
setsid gnome-calculator >/tmp/calc.log 2>&1 &
sleep 5
E
bash start.sh
. ./envfile
agent-desktop status

### 설치 버전과 접근성 권한·세션 상태 확인

In [ ]:
%%bash
cd /work; . ./envfile
agent-desktop version
echo
agent-desktop status | head -c 600; echo

### snapshot -i로 버튼 ref 목록 보기

In [ ]:
%%bash
cd /work; . ./envfile
mkdir -p outputs
agent-desktop snapshot --app gnome-calculator -i --compact > s1.json
python3 - <<'E'
import json
d=json.load(open('/work/s1.json'))['data']
print(d['snapshot_id'],d['ref_count'])
def walk(n):
    if 'ref_id' in n: yield n
    for c in n.get('children',[]): yield from walk(c)
for n in walk(d['tree']):
    print(n['ref_id'],n['role'],repr(n.get('name')),n.get('available_actions'))
E

### 7 + 3 = 클릭 후 결과값 확인

In [ ]:
%%bash
cd /work; . ./envfile
R(){ python3 -c "
import json
d=json.load(open('/work/s1.json'))['data']
def w(n):
    if 'ref_id' in n: yield n
    for c in n.get('children',[]): yield from w(c)
for n in w(d['tree']):
    if n.get('name')=='$1': print(n['ref_id'])
"; }
import -window root outputs/input_calc_before_fullscreen.png
for k in "7 7" "+ +" "3 3" "= ="; do r=$(R "$k"); echo "== $k $r"; agent-desktop click "$r"; echo; done
import -window root outputs/result_calc_after_fullscreen.png
agent-desktop snapshot --app gnome-calculator -i --compact > s2.json
python3 - <<'E'
import json
d=json.load(open('/work/s2.json'))['data']
def w(n):
    yield n
    for c in n.get('children',[]): yield from w(c)
for n in w(d['tree']):
    if n.get('role')=='textfield': print(n['ref_id'], n.get('value'))
E

### STALE_REF 만들기 (계산기 재시작 후 옛 ref 클릭)

In [ ]:
%%bash
cd /work; . ./envfile
OLD=$(python3 -c "import json;print(json.load(open('/work/s1.json'))['data']['snapshot_id'])")
agent-desktop click @$OLD:e99; echo
pkill -x gnome-calculato; sleep 1
GTK_MODULES=gail:atk-bridge GNOME_ACCESSIBILITY=1 setsid gnome-calculator >/dev/null 2>&1 &
sleep 5
agent-desktop click @$OLD:e14; echo

### screenshot 찍기

In [ ]:
%%bash
cd /work; . ./envfile
agent-desktop screenshot --app gnome-calculator outputs/ad_screenshot.png
identify outputs/ad_screenshot.png

### 접근성 버스에 없는 앱 xeyes

In [ ]:
%%bash
cd /work; . ./envfile
setsid xeyes >/dev/null 2>&1 &
sleep 2
agent-desktop snapshot --app xeyes -i --compact | head -c 700; echo
agent-desktop list-windows | python3 -c "import json,sys;print([(w['app_name'],w['accessible']) for w in json.load(sys.stdin)['data']])"